# Лабораторная работа №3 — RAG по PDF

**Источник:** загруженный PDF с условием лабораторной работы. Индексируются содержательные страницы 3–12 (титул и оглавление не включены).

**Статус:** реальные офлайн-эксперименты поиска выполнены; результат получен на TF-IDF + SVD, а не OllamaEmbeddings. Ответы ниже — детерминированные цитаты из контекста, **не ответы LLM**. Для выполнения с настоящей LLM предусмотрена отдельная ячейка с Ollama.

## 0. Установка и конфигурация

In [1]:
# Для Colab: загрузите рядом файлы rag_lab3.py и исходный PDF
# !pip -q install pymupdf scikit-learn pandas numpy requests
from pathlib import Path
from rag_lab3 import (load_pages, split_pages, dense_index, lexical_index, rank,
                      hit, grounded_answer, get_all_results, ollama_answer, QA,
                      NEGATIVE, BASE_PROMPT, STRICT_PROMPT)
print('PDF существует:', Path('Практическая_работа__3_Совр_Инт_Сет_Сер.pdf').exists())

PDF существует: True


## 1. Базовый RAG-конвейер

In [ ]:
pages = load_pages()
corpus = pages[2:]
chunks = split_pages(corpus, 1000, 200)
index = dense_index(chunks)
q = QA[0]
retrieved = rank(q['question'], chunks, idx=index, k=3)
print('Страниц:', len(pages), 'Индексируемых:', len(corpus), 'Чанков:', len(chunks))
print('Исходный запрос:', q['question'])
for x in retrieved: print(f"Фрагмент {x['id']}, стр. {x['page']}, позиция {x['start_index']}, score={x['score']}:", x['text'][:130])
print('Ответ:', grounded_answer(q['question'], retrieved, q))

## 2. Эксперименты 1–7 с сохранёнными результатами

In [3]:
tables, metadata = get_all_results()
print('Выполнено экспериментов:', len(tables))
print('Файлы результатов:', sorted(str(p) for p in Path('results_lab3').glob('*.csv')))

Выполнено экспериментов: 7
Файлы результатов: ['results_lab3/task1.csv', 'results_lab3/task2.csv', 'results_lab3/task3.csv', 'results_lab3/task4.csv', 'results_lab3/task5.csv', 'results_lab3/task6.csv', 'results_lab3/task7.csv']


### Задание 1. Полнота индекса

In [4]:
print(tables['task1'].to_string(index=False, max_colwidth=80))

question  variant  group   hit         ids      pages
     Q01 Первые 5 начало  True   [5, 4, 1]  [4, 4, 3]
     Q01   Полный начало False  [25, 5, 6] [10, 4, 4]
     Q02 Первые 5 начало False   [5, 1, 2]  [4, 3, 3]
     Q02   Полный начало False [22, 21, 6]  [9, 9, 4]
     Q03 Первые 5 начало  True   [5, 1, 2]  [4, 3, 3]
     Q03   Полный начало  True  [5, 6, 19]  [4, 4, 8]
     Q04 Первые 5  далее False   [1, 2, 4]  [3, 3, 4]
     Q04   Полный  далее  True [18, 8, 13]  [8, 5, 6]


### Задание 2. Размер фрагментов

In [5]:
print(tables['task2'].to_string(index=False, max_colwidth=80))

 size  chunks question   hit          ids
  500      46      Q01 False  [40, 7, 13]
  500      46      Q02 False  [34, 33, 8]
  500      46      Q03  True    [7, 9, 8]
  500      46      Q04  True  [12, 31, 9]
  500      46      Q05  True  [13, 6, 18]
  500      46      Q06  True  [18, 35, 2]
  500      46      Q07  True [42, 20, 39]
  500      46      Q08  True [25, 34, 24]


### Задание 3. Перекрытие

In [6]:
print(tables['task3'].to_string(index=False, max_colwidth=80))

 overlap  chunks question   hit  has_near_duplicates          ids
       0      25      Q04  True                False  [15, 6, 16]
       0      25      Q05  True                False    [7, 9, 3]
       0      25      Q06  True                False  [9, 18, 16]
       0      25      Q07  True                False [10, 23, 21]
       0      25      Q08 False                False [13, 15, 18]
     100      27      Q04  True                False  [7, 16, 17]
     100      27      Q05  True                False  [8, 10, 11]
     100      27      Q06  True                False [10, 20, 17]


### Задание 4. Бинарный мешок слов против плотного SVD

In [7]:
print(tables['task4'].to_string(index=False, max_colwidth=80))

             method       phrasing question   hit          ids       pages
Бинарный мешок слов         точный      Q01 False [25, 21, 14]  [10, 9, 6]
Бинарный мешок слов перефразировка      Q01 False  [25, 26, 9] [10, 10, 5]
Бинарный мешок слов         точный      Q02 False  [21, 22, 6]   [9, 9, 4]
Бинарный мешок слов перефразировка      Q02 False  [5, 12, 17]   [4, 6, 7]
Бинарный мешок слов         точный      Q03  True   [5, 20, 3]   [4, 8, 3]
Бинарный мешок слов перефразировка      Q03 False    [8, 6, 7]   [5, 4, 4]
Бинарный мешок слов         точный      Q04 False  [20, 14, 2]   [8, 6, 3]
Бинарный мешок слов перефразировка      Q04 False    [5, 4, 7]   [4, 4, 4]


### Задание 5. Проверка ссылок

In [8]:
print(tables['task5'].to_string(index=False, max_colwidth=80))

question  cited  citation_exists  supported  page_correct
     Q01  False            False      False         False
     Q02  False            False      False         False
     Q03   True             True       True          True
     Q04   True             True       True          True
     Q05   True             True       True          True


### Задание 6. Положительные / отрицательные вопросы и два промпта

In [9]:
print(tables['task6'].to_string(index=False, max_colwidth=80))

  prompt question  answer_in_pdf  retrieval_hit  grade_proxy  correct_refusal  unsupported
Исходный      Q01           True          False          0.0            False        False
 Строгий      Q01           True          False          0.0            False        False
Исходный      Q02           True          False          0.0            False        False
 Строгий      Q02           True          False          0.0            False        False
Исходный      Q03           True           True          2.0            False        False
 Строгий      Q03           True           True          2.0            False        False
Исходный      Q04           True           True          2.0            False        False
 Строгий      Q04           True           True          2.0            False        False


### Задание 7. Уточняющие вопросы и сессии

In [10]:
print(tables['task7'].to_string(index=False, max_colwidth=80))

 session  step                       question                          query  history_len   hit  grade_proxy retrieved_ids retrieved_pages
       1     1                 Что такое RAG?                 Что такое RAG?            0 False            0    [25, 5, 6]      [10, 4, 4]
       1     2      Как этот подход работает? Как работает подход RAG с п...            1  True            2    [4, 21, 1]       [4, 9, 3]
       1     3    Изменяет ли он веса модели? Изменяет ли RAG веса языков...            2  True            2    [4, 1, 18]       [4, 3, 8]
       2     1 Что такое бинарный мешок слов? Что такое бинарный мешок слов?            0 False            0   [22, 21, 6]       [9, 9, 4]
       2     2    Что в нём означает единица? Что означает значение 1 в б...            1 False            0     [5, 8, 6]       [4, 5, 4]
       2     3  Учитывает ли он порядок слов? Учитывает ли бинарный мешок...            2 False            0    [5, 6, 22]       [4, 4, 9]
       3     1      Что так

## 3. Настоящая генерация с Ollama (запуск при доступном локальном сервере)

In [ ]:
# В Colab локальный Ollama недоступен без настройки сервера/туннеля.
# Локально запустите: ollama pull qwen2.5:3b
# Затем: ollama serve
# Сравнение обоих промптов проводится на неизменных top-3 фрагментах.
RUN_OLLAMA = False
if RUN_OLLAMA:
    for q in QA[:5]:
        contexts = rank(q['question'], chunks, idx=index, k=3)
        for strict in (False, True):
            response = ollama_answer(q['question'], contexts, strict=strict, model='qwen2.5:3b')
            print(q['id'], 'строгий' if strict else 'исходный', response)
    for question in NEGATIVE:
        contexts = rank(question, chunks, idx=index, k=3)
        for strict in (False, True):
            print(question, strict, ollama_answer(question, contexts, strict=strict))

## 4. Пример диалога с самостоятельным запросом и отдельными сессиями

In [ ]:
sessions = {}
def ask(session_id, question, standalone_query):
    history = sessions.setdefault(session_id, [])
    contexts = rank(standalone_query, chunks, idx=index, k=3)
    result = {'session':session_id, 'original_question':question,
              'search_query':standalone_query,
              'sources':[(x['id'],x['page']) for x in contexts],
              'response':grounded_answer(standalone_query,contexts,next((q for q in QA if q['question']==standalone_query),None)),
              'history_used':history[-3:]}
    history.append({'question':question,'answer':result['response']})
    return result
print(ask('conversation-1','Что такое RAG?','Что такое RAG?'))
print(ask('conversation-1','А как он работает?','Как работает RAG?'))
print(ask('conversation-2','Что такое TF-IDF?','Что такое TF-IDF?'))
assert len(sessions['conversation-1'])==2 and len(sessions['conversation-2'])==1
sessions['conversation-1'].clear()
assert len(sessions['conversation-1'])==0

## 5. Ограничения и воспроизводимость

- Hit@3 оценивается наличием заранее выбранной подтверждающей подстроки на эталонной странице в первых трёх фрагментах. Это **строгий прокси**, а не полная экспертная разметка достаточности.
- Разница между исходным и строгим промптом **не оценена экспериментально на LLM** без запуска Ollama; офлайн-экстрактор имеет одну политику отказа.
- SVD — низкоразмерное плотное математическое представление, но **не модель нейросетевых эмбеддингов**. Для точного выполнения задания 4 нужна работающая OllamaEmbeddings.
- Для внешних диалогов необходимо генерировать переформулировки автоматически; демонстрационные запросы подготовлены заранее и сохраняются в CSV.
- Программные версии записаны в `results_lab3/summary.json`.